# Testing Model Outputs
Alex has run some inferences we now need to test to see if they match what is already produced and available in the fafb dataset

In [1]:
from load_data.connect_clients import connect_cave_client, connect_flywire_client #Note move this to the main repo level to run, as a module
ENV_PATH = ".env"

# Connect to CAVE client
cave_client = connect_cave_client(ENV_PATH)

# Connect to FlyWire client
flywire_client = connect_flywire_client(ENV_PATH)

/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CAVE token already exists in your account. No need to add it again.
FLYWIRE token already exists in your account. No need to add it again.


## Load in JSON file

In [3]:
test_json = "./data/skeleton_test_t1_p1_fixed.json"
# Load in JSON file as a dataframe
import pandas as pd
import json

with open(test_json, 'r') as file:
    data = json.load(file)

df = pd.DataFrame(data)
display(df.head())

,_id,synapse_id,prediction
0,{'$oid': '6800c80f380cc9ed5a879215'},98373,"[2.1970388843328692e-05, 0.9999692440032959, 9..."
1,{'$oid': '6800c80f380cc9ed5a879216'},98415,"[2.5887402443913743e-05, 0.9999715089797974, 2..."
2,{'$oid': '6800c80f380cc9ed5a879217'},99167,"[5.943149062659359e-06, 0.9999641180038452, 1...."
3,{'$oid': '6800c80f380cc9ed5a879218'},99242,"[9.46293948800303e-06, 0.9999514818191528, 3.5..."
4,{'$oid': '6800c80f380cc9ed5a879219'},304977,"[0.028452927246689796, 0.8848700523376465, 0.0..."


## Load in json input file

# Check out Cave Client tables


In [9]:
cave_client.materialize.get_tables()

['hierarchical_neuron_annotations',
 'neuron_information_v2',
 'synapses_nt_v1',
 'nuclei_v1',
 'proofread_neurons',
 'fly_synapses_neuropil_v6']

In [30]:
# "x":438817,"y":164242,"z":217400

centre_point = [438817, 164242, 217400]
min_bounding_box = [438817 - 1000, 164242 - 1000, 217400 - 1000]
max_bounding_box = [438817 + 1000, 164242 + 1000, 217400 + 1000]
bounding_box = [min_bounding_box, max_bounding_box]

synapse_table = cave_client.info.get_datastack_info('synapses_nt_v1')
# df=cave_client.materialize.query_table(synapse_table,
#                                   filter_spatial_dict = {'post_pt_position': bounding_box})
# 
# 
# df=cave_client.materialize.synapse_query(
#                                   bounding_box=bounding_box)
# df
cave_client.materialize.synapse_query("synapses_nt_v1"
                                      )

HTTPError: 400 Client Error: BAD REQUEST for url: https://global.daf-apis.com/info/api/v2/datastack/full/synapses_nt_v1 content: b'{\n  "data": {\n    "auth_dataset": null,\n    "resource_namespace": "datastack",\n    "table_id": "synapses_nt_v1"\n  },\n  "error": "invalid_table_id",\n  "message": "Invalid table_id for service"\n}\n'

In [19]:
cave_client.materialize.get_tables()

['hierarchical_neuron_annotations',
 'neuron_information_v2',
 'synapses_nt_v1',
 'nuclei_v1',
 'proofread_neurons',
 'fly_synapses_neuropil_v6']

In [20]:
cave_client.materialize.get_table_metadata(
    table_name='synapses_nt_v1',
)

{'schema': 'fly_nt_synapse',
 'id': 7470,
 'table_name': 'synapses_nt_v1',
 'valid': True,
 'created': '2021-03-09T20:14:58.183080',
 'aligned_volume': 'fafb_seung_alignment_v0',
 'schema_type': 'fly_nt_synapse',
 'user_id': 'foo@bar.com',
 'description': 'FlyWire synapse description\r\nSynapse version: 20191211\r\nNT version: 20201223\r\n\r\nSynapses in this table consist of a pre- and a postsynaptic point (in nm), confidence scores, and neurotransmitter information. \r\nThe synapses were predicted by Buhmann et al [1] for the v14 alignment of the FAFB dataset. The FlyWire team remapped these synapses into the v14.1 space used by FlyWire with an accuracy of <64nm (therefore, this is a potential source of error). This version of the Buhmann et al. synapses was trained on the initial training set from the calyx and performance varies across brain areas accordingly.\r\nBuhmann et al. assigned two scores to their synapses representing different measurements of confidence. The “connection_

# View a synapse

In [31]:
from cloudvolume import CloudVolume
import numpy as np
import daisy
import napari

def validate_locs_nm(locs_nm, voxel_size, volume_shape_vox, crop_size_vox):
    """
    Validate and filter `locs_nm` to ensure they produce valid crops within the volume.

    Args:
        locs_nm (list of tuples): Locations in nanometers (Z, Y, X).
        voxel_size (tuple): Voxel size in nm (Z, Y, X).
        volume_shape_vox (tuple): Shape of the CloudVolume in voxels (X, Y, Z).
        crop_size_vox (tuple): Size of the crop in voxels (Z, Y, X).

    Returns:
        list: Validated and filtered `locs_nm` within volume bounds.
    """
    valid_locs = []

    # Convert volume shape to nanometers
    volume_extent_nm = tuple(vs * sz for vs, sz in zip(voxel_size[::-1], volume_shape_vox))

    crop_half_nm = tuple((cs * vs) // 2 for cs, vs in zip(crop_size_vox, voxel_size))

    for loc in locs_nm:
        z_nm, y_nm, x_nm = loc

        in_bounds = (
            crop_half_nm[0] <= z_nm < (volume_extent_nm[2] - crop_half_nm[0]) and
            crop_half_nm[1] <= y_nm < (volume_extent_nm[1] - crop_half_nm[1]) and
            crop_half_nm[2] <= x_nm < (volume_extent_nm[0] - crop_half_nm[2])
        )

        if in_bounds:
            valid_locs.append(loc)
        else:
            print(f"Skipping out-of-bounds loc (nm): {loc}")

    return valid_locs


def get_fafb_v14_voxels(
    locs_nm,
    voxel_size=(40, 4, 4),  # CloudVolume native voxel size
    size=(16, 160, 160),    # in voxel units
    precomputed_path="https://storage.googleapis.com/neuroglancer-fafb-data/fafb_v14/fafb_v14_orig/",
):
    vol = CloudVolume(precomputed_path, mip=0, cache=True, parallel=False)
    size = daisy.Coordinate(size)
    shape = vol.shape  # (X, Y, Z)

    # Convert locations in nanometers to voxel coordinates at native resolution
    locs_vox = [
        (
            int(z_nm / voxel_size[0]),
            int(y_nm / voxel_size[1]),
            int(x_nm / voxel_size[2])
        )
        for (z_nm, y_nm, x_nm) in locs_nm
    ]

    raw = []

    for loc in locs_vox:
        loc = daisy.Coordinate(loc)  # Z, Y, X
        start = loc - (size / 2)
        roi = daisy.Roi(start, size)

        z0, y0, x0 = map(int, roi.get_begin())
        sz, sy, sx = map(int, size)
        z1, y1, x1 = z0 + sz, y0 + sy, x0 + sx

        # Convert to CloudVolume order → X, Y, Z
        x0c, y0c, z0c = x0, y0, z0
        x1c, y1c, z1c = x1, y1, z1

        if not (
            0 <= x0c < shape[0] and x1c <= shape[0] and
            0 <= y0c < shape[1] and y1c <= shape[1] and
            0 <= z0c < shape[2] and z1c <= shape[2]
        ):
            print(f"Out-of-bounds: loc={loc}, range=({x0c}:{x1c}, {y0c}:{y1c}, {z0c}:{z1c})")
            chunk = np.zeros((sz, sy, sx), dtype=np.uint8)
        else:
            try:
                chunk = vol[x0c:x1c, y0c:y1c, z0c:z1c]
                chunk = np.asarray(chunk)

                if chunk.ndim == 4 and chunk.shape[-1] == 1:
                    chunk = np.squeeze(chunk, axis=-1)

                chunk = chunk.transpose(2, 1, 0)  # (Z, Y, X)
            except Exception as e:
                print(f"Error at {loc}: {e}")
                chunk = np.zeros((sz, sy, sx), dtype=np.uint8)

        raw.append(chunk.astype(np.float32))

    raw = np.stack(raw)
    raw_normalized = raw / 255.0 * 2.0 - 1.0
    return raw, raw_normalized


# Set voxel size and crop size
voxel_size = (40, 4, 4)
crop_size_vox = (16, 160, 160)

# Example locations (some may be invalid)
#z y x
locs_nm = [(214080, 162452, 437042), (214400, 163690, 436761), (212360, 160857, 438812), (213200, 160988, 437477), (206240, 142887, 451693), (206480, 142469, 452441), (203200, 127946, 417736), (219920, 148945, 437094), (215480, 161447, 438157), (216440, 165325, 435448), (219800, 145680, 438400), (190120, 135854, 449539), (190680, 135110, 451904), (178880, 138534, 461270), (219560, 145662, 439271), (208520, 157703, 444303), (207240, 141157, 451531), (215280, 165339, 435122), (220320, 148466, 440063), (206160, 141130, 438888), (199240, 127005, 418373), (187520, 130316, 456325), (206640, 141202, 450800), (208360, 143732, 437101), (205600, 142319, 437770), (187840, 133394, 450664), (187960, 134227, 449626), (181720, 136846, 465244), (199880, 127893, 419062), (199680, 129860, 419688), (220840, 148560, 439079), (221960, 147025, 439903), (219960, 146385, 437533), (195720, 129139, 423619), (202640, 128987, 417726), (181280, 137342, 462296), (199560, 127105, 416967), (209520, 159910, 441919), (209600, 158477, 441664), (208480, 157681, 442617), (188240, 160057, 343529), (164040, 161445, 341698), (184640, 159803, 343507), (175440, 162142, 349611)]

# Validate locations
# Get data
raw, raw_norm = get_fafb_v14_voxels(
    locs_nm=locs_nm,
    voxel_size=voxel_size,
    size=crop_size_vox
)
np.set_printoptions(threshold=np.inf)
print("Shape:", raw.shape)


print("Sample voxel [0,0,0,0]:", raw[0, 0, 0, 0])
print("Slice [0, Z mid]:")
print(raw[0, raw.shape[1] // 2])

ModuleNotFoundError: No module named 'daisy'

## Trial 2: Using flywire

In [4]:
# Import skeleton ids
pth = "./data/test_data_from_alex/skeletons_fixed.json"
skeleton_df = pd.read_json(pth)
skeleton_df

,_id,skeleton_id,hemi_lineage_id,nt_known
0,{'$oid': '6800ba442b847cf645a963c1'},16,1,[acetylcholine]
1,{'$oid': '6800ba442b847cf645a963c2'},27,1,[acetylcholine]
2,{'$oid': '6800ba442b847cf645a963c3'},430,2,None
3,{'$oid': '6800ba442b847cf645a963c4'},734,3,[acetylcholine]
4,{'$oid': '6800ba442b847cf645a963c5'},949,1,[acetylcholine]
...,...,...,...,...
2799,{'$oid': '6800ba442b847cf645a96eb0'},11150632,90,[dopamine]
2800,{'$oid': '6800ba442b847cf645a96eb1'},11175371,90,[dopamine]
2801,{'$oid': '6800ba442b847cf645a96eb2'},11266651,90,[dopamine]
2802,{'$oid': '6800ba442b847cf645a96eb3'},11267636,90,[dopamine]


In [5]:
skeleton_ids = skeleton_df['skeleton_id'].tolist()
print("Skeleton IDs:", skeleton_ids)

Skeleton IDs: [16, 27, 430, 734, 949, 1165, 2076, 2115, 3133, 5714, 12578, 21999, 22132, 22277, 22422, 22594, 22744, 22906, 22976, 23005, 23134, 23432, 23512, 23569, 23597, 23829, 24251, 24622, 24726, 27048, 27246, 27295, 27611, 28876, 30434, 30571, 30791, 30891, 32214, 32399, 32793, 32801, 33903, 35246, 35447, 36108, 36390, 37212, 37235, 37250, 37935, 38885, 39139, 39254, 39668, 39682, 40306, 40637, 40749, 41308, 41578, 42421, 42927, 43539, 45242, 46493, 46800, 49026, 49865, 51080, 51886, 52106, 53631, 53671, 54072, 55085, 55125, 56424, 56623, 56983, 56995, 56999, 57003, 57007, 57011, 57015, 57019, 57023, 57035, 57039, 57047, 57051, 57059, 57063, 57067, 57071, 57076, 57080, 57089, 57094, 57098, 57102, 57106, 57114, 57122, 57126, 57130, 57134, 57138, 57142, 57146, 57154, 57158, 57166, 57171, 57175, 57179, 57192, 57196, 57200, 57204, 57208, 57212, 57216, 57220, 57224, 57232, 57236, 57241, 57246, 57254, 57258, 57266, 57270, 57274, 57278, 57307, 57311, 57319, 57323, 57333, 57337, 57341, 5

In [8]:
from fafbseg import flywire
import pymaid

tk = flywire.get_chunkedgraph_secret()
# Connect to the VFB's CATMAID
rm = pymaid.CatmaidInstance('https://fafb.catmaid.virtualflybrain.org/',
                            project_id=1, api_token=None)

root_ids = flywire.skid_to_id([16])
print("Root IDs:", root_ids)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)






Root IDs:   skeleton_id          flywire_id  confidence
0          16  720575940636873791        0.97


In [10]:
root = root_ids.iloc[0]['flywire_id']
print(root)

720575940636873791


In [11]:
# rt = str(root_id['flywire_id'].values)
rt = str(root)
pre_df = cave_client.materialize.query_table(
    table='synapses_nt_v1',
    filter_in_dict={'pre_pt_root_id': [rt]},
)

In [12]:
pre_df

,id,created,superceded_id,valid,connection_score,cleft_score,gaba,ach,glut,oct,ser,da,valid_nt,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,pre_pt_position,post_pt_position
0,139694309,2021-03-09 20:14:58.183080+00:00,NaN,t,74.095100,138,2.987464e-06,0.999997,3.978493e-13,1.721913e-09,1.470456e-12,8.605384e-08,t,78111643170875923,720575940636873791,78111643170864308,720575940618376327,"[436588, 164940, 214640]","[436500, 164872, 214640]"
1,34146983,2021-03-09 20:14:58.183080+00:00,NaN,t,107.203735,0,2.513170e-03,0.000102,4.041771e-06,4.571922e-01,4.108277e-01,1.293604e-01,t,79097148918404779,720575940636873791,79097148918404737,720575940630580183,"[491324, 185564, 135160]","[491216, 185496, 135160]"
2,110296150,2021-03-09 20:14:58.183080+00:00,NaN,t,17.316696,64,6.559769e-05,0.999764,8.223993e-08,4.060483e-05,2.598249e-08,1.298157e-04,t,77196505765110253,720575940636873791,77196505765110238,720575940645686820,"[382392, 142632, 179080]","[382432, 142636, 179120]"
3,96755819,2021-03-09 20:14:58.183080+00:00,NaN,t,6.669370,131,2.813351e-02,0.567284,1.525789e-02,1.500118e-03,3.144438e-01,7.338025e-02,t,78535023263201092,720575940636873791,78535023263202313,720575940627605779,"[461968, 233612, 35480]","[462064, 233508, 35480]"
4,114074538,2021-03-09 20:14:58.183080+00:00,NaN,t,331.848938,144,4.737731e-07,0.999983,1.063264e-12,5.372817e-06,3.894931e-12,1.144057e-05,t,76633899408897733,720575940636873791,76633899408897799,720575940626312835,"[348896, 165992, 173560]","[348888, 166120, 173520]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33870,19545410,2021-03-09 20:14:58.183080+00:00,NaN,t,146.762360,150,6.641940e-07,0.988201,1.454832e-11,1.179557e-02,3.861234e-12,2.395146e-06,t,76633968128168475,720575940636873791,76633968128177025,720575940409968521,"[348448, 166988, 167360]","[348512, 167112, 167400]"
33871,19545413,2021-03-09 20:14:58.183080+00:00,NaN,t,99.041374,141,1.716046e-03,0.908339,7.988337e-04,8.491956e-02,3.502868e-05,4.191289e-03,t,76563599383985534,720575940636873791,76563599384000366,720575940414393546,"[347648, 166768, 167440]","[347520, 166760, 167440]"
33872,19545416,2021-03-09 20:14:58.183080+00:00,NaN,t,13.393262,137,1.420828e-06,0.979336,1.319826e-10,2.065772e-02,2.105955e-11,5.182661e-06,t,76633968128168475,720575940636873791,76633968128177025,720575940409968521,"[348436, 166968, 167360]","[348488, 167132, 167440]"
33873,31527073,2021-03-09 20:14:58.183080+00:00,NaN,t,163.185516,154,2.809199e-02,0.971633,1.117512e-05,8.874081e-06,8.303835e-07,2.542115e-04,t,78392705830373477,720575940636873791,78392705830461565,720575940604009452,"[450940, 139892, 206120]","[450876, 139976, 206160]"


In [ ]:
# Split 

In [50]:
# Coordinates for the synapse
xyz = [438817, 164242, 217400]

# Convert xyz to tuple for comparison, since pre_pt_position is a list
view = pre_df[pre_df['post_pt_position'].apply(lambda pos: list(pos) == xyz)]
print("View DataFrame:")
print(view)

View DataFrame:
Empty DataFrame
Columns: [id, created, superceded_id, valid, connection_score, cleft_score, gaba, ach, glut, oct, ser, da, valid_nt, pre_pt_supervoxel_id, pre_pt_root_id, post_pt_supervoxel_id, post_pt_root_id, pre_pt_position, post_pt_position]
Index: []


## Predictions for single test neuron

In [16]:
input_df = pd.read_parquet("./data/test_data_from_alex/single_neuron/test_root_presynapses.parquet")
display(input_df.head())

,root_id,post_synaptic_root_id,xyz_voxel,supervoxel_id,voxel_resolution_nm
0,720575940624928574,720575940619434712,"[672464, 123968, 128240]",82192342869822207,"[1.0, 1.0, 1.0]"
1,720575940624928574,720575940631766339,"[722744, 172224, 124160]",83037592433525500,"[1.0, 1.0, 1.0]"
2,720575940624928574,720575940637335262,"[693408, 125320, 113120]",82544186523740614,"[1.0, 1.0, 1.0]"
3,720575940624928574,720575940627196691,"[716104, 138956, 142000]",82896305189929007,"[1.0, 1.0, 1.0]"
4,720575940624928574,720575940622726271,"[709816, 136728, 142040]",82825867726245398,"[1.0, 1.0, 1.0]"


In [22]:
file_path = "./data/test_data_from_alex/single_neuron/19000101_000000_predictions.json"
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():  # Skip empty lines
            data.append(line.strip())
print(data)

['[', '{', '"split_name": "skeleton",', '"prediction": [', '0.04386502876877785,', '0.0024406160227954388,', '0.02535090036690235,', '0.016086909919977188,', '1.2697292731900234e-05,', '0.9122439026832581', '],', '"experiment": "test",', '"train_number": 1,', '"predict_number": 1,', '"x": 563932,', '"y": 171384,', '"z": 74560', '},', '{', '"split_name": "skeleton",', '"prediction": [', '0.22232316434383392,', '0.022955579683184624,', '0.045275382697582245,', '0.015366428531706333,', '4.4973206968279555e-05,', '0.6940343976020813', '],', '"experiment": "test",', '"train_number": 1,', '"predict_number": 1,', '"x": 554056,', '"y": 156480,', '"z": 75640', '},', '{', '"split_name": "skeleton",', '"prediction": [', '0.01213825959712267,', '0.9082674980163574,', '0.012392234988510609,', '0.0006509912782348692,', '0.014343288727104664,', '0.05220772698521614', '],', '"experiment": "test",', '"train_number": 1,', '"predict_number": 1,', '"x": 437656,', '"y": 248080,', '"z": 92240', '},', '{', '

In [35]:
# Convert JSON to dictionary
import json
file_path = "./data/test_data_from_alex/single_neuron/19000101_000000_predictions.json"

json_data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():  # Skip empty lines
            json_data.append(line)
# Check the data
print(f"Number of entries: {len(json_data)}")
# Convert to dictionary
output_df = pd.DataFrame(json_data)
output_df.head()



Number of entries: 81543


,0
0,[\n
1,{\n
2,"""split_name"": ""skeleton"",\n"
3,"""prediction"": [\n"
4,"0.04386502876877785,\n"


In [33]:
json_data

['[\n',
 '  {\n',
 '    "split_name": "skeleton",\n',
 '    "prediction": [\n',
 '      0.04386502876877785,\n',
 '      0.0024406160227954388,\n',
 '      0.02535090036690235,\n',
 '      0.016086909919977188,\n',
 '      1.2697292731900234e-05,\n',
 '      0.9122439026832581\n',
 '    ],\n',
 '    "experiment": "test",\n',
 '    "train_number": 1,\n',
 '    "predict_number": 1,\n',
 '    "x": 563932,\n',
 '    "y": 171384,\n',
 '    "z": 74560\n',
 '  },\n',
 '  {\n',
 '    "split_name": "skeleton",\n',
 '    "prediction": [\n',
 '      0.22232316434383392,\n',
 '      0.022955579683184624,\n',
 '      0.045275382697582245,\n',
 '      0.015366428531706333,\n',
 '      4.4973206968279555e-05,\n',
 '      0.6940343976020813\n',
 '    ],\n',
 '    "experiment": "test",\n',
 '    "train_number": 1,\n',
 '    "predict_number": 1,\n',
 '    "x": 554056,\n',
 '    "y": 156480,\n',
 '    "z": 75640\n',
 '  },\n',
 '  {\n',
 '    "split_name": "skeleton",\n',
 '    "prediction": [\n',
 '    

In [25]:
output_df = pd.DataFrame(data)
output_df.head()

,0
0,[
1,{
2,"""split_name"": ""skeleton"","
3,"""prediction"": ["
4,"0.04386502876877785,"
